In [1]:
%pip install fairlearn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\ghate\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GroupKFold,
    cross_validate, learning_curve
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    classification_report, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from fairlearn.metrics import MetricFrame, selection_rate, \
    demographic_parity_difference, equalized_odds_difference

RANDOM_STATE = 42

retail = pd.read_excel('online_retail_II.xlsx')  
retail['is_cancelled'] = retail['Invoice'].astype(str).str.startswith('C').astype(int)
retail = retail.drop('Customer ID', axis=1, errors='ignore')
retail = retail.drop('Description', axis=1, errors='ignore')
retail['month'] = retail['InvoiceDate'].dt.month
retail['day_of_week_num'] = retail['InvoiceDate'].dt.dayofweek
retail['hour'] = retail['InvoiceDate'].dt.hour
retail = retail[retail['StockCode'] != 'B']
retail = retail[retail['Price'] != 0]
retail['Quantity'] = retail['Quantity'].abs()

top_countries = retail['Country'].value_counts().nlargest(6).index
retail['Country_grouped'] = retail['Country'].where(retail['Country'].isin(top_countries), 'Other')
country_grouped_full = retail['Country_grouped'].copy()  # kept aside for Fairlearn

retail_dum = pd.get_dummies(retail, columns=['Country_grouped'], drop_first=True)
retail_dum = retail_dum.drop('Country', axis=1)
retail_dum = retail_dum.drop(['Invoice', 'StockCode', 'InvoiceDate'], axis=1)

X = retail_dum.drop('is_cancelled', axis=1)
y = retail_dum['is_cancelled']

#Stratified 5-fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'f1': 'f1', 'precision': 'precision', 'recall': 'recall'}

clf = DecisionTreeClassifier(random_state=RANDOM_STATE)
cv_results = cross_validate(clf, X, y, cv=skf, scoring=scoring)
print("=== Decision Tree, stratified 5-fold CV ===")
for k in scoring:
    vals = cv_results[f'test_{k}']
    print(f"{k}: mean={vals.mean():.4f} std={vals.std():.4f}")

clf_bal = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced')
cv_results_bal = cross_validate(clf_bal, X, y, cv=skf, scoring=scoring)
print("\n=== Decision Tree (balanced), stratified 5-fold CV ===")
for k in scoring:
    vals = cv_results_bal[f'test_{k}']
    print(f"{k}: mean={vals.mean():.4f} std={vals.std():.4f}")

#Learning curve for the classifier
train_sizes, train_scores, test_scores = learning_curve(
    clf, X, y, cv=skf, scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
)
plt.figure(figsize=(6, 4.2))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training F1')
plt.plot(train_sizes, test_scores.mean(axis=1), 'o-', label='Cross-validation F1')
plt.xlabel('Training set size (transactions)')
plt.ylabel('F1-score (cancellation class)')
plt.title('Learning Curve: Decision Tree Classifier')
plt.legend()
plt.tight_layout()
plt.savefig('learning_curve_classifier.png', dpi=150)
plt.close()

#Fairlearn bias audit 
X_train, X_test, y_train, y_test, cg_train, cg_test = train_test_split(
    X, y, country_grouped_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

mf = MetricFrame(
    metrics={'selection_rate': selection_rate, 'precision': precision_score,
             'recall': recall_score, 'f1': f1_score},
    y_true=y_test, y_pred=y_pred, sensitive_features=cg_test
)
print("\n=== Fairlearn MetricFrame by Country ===")
print(mf.by_group)
print("\nDemographic parity difference:",
      demographic_parity_difference(y_test, y_pred, sensitive_features=cg_test))
print("Equalized odds difference:",
      equalized_odds_difference(y_test, y_pred, sensitive_features=cg_test))

#Walmart data
walmart = pd.read_csv('Walmart.csv')
scaler = StandardScaler()
numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
walmart_scaled = walmart.copy()
walmart_scaled[numeric_features] = scaler.fit_transform(walmart[numeric_features])

walmart_dummies = pd.get_dummies(walmart_scaled, columns=['Store'], drop_first=True)
store_cols = [c for c in walmart_dummies.columns if c.startswith('Store_')]
X2 = walmart_dummies[['Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment'] + store_cols]
y2 = walmart_dummies['Weekly_Sales']
groups = walmart['Store']

# 6. Group 5-fold CV for kNN Regressor 
gkf = GroupKFold(n_splits=5)
reg_scoring = {'r2': 'r2', 'neg_mae': 'neg_mean_absolute_error', 'neg_rmse': 'neg_root_mean_squared_error'}
knn = KNeighborsRegressor(n_neighbors=5)
cv_reg = cross_validate(knn, X2, y2, cv=gkf, groups=groups, scoring=reg_scoring)
print("\n=== kNN Regressor (store one-hot), GroupKFold by Store ===")
for k in reg_scoring:
    vals = cv_reg[f'test_{k}']
    if 'neg' in k:
        vals = -vals
    print(f"{k}: mean={vals.mean():.4f} std={vals.std():.4f}")

#Learning curve for regressor
train_sizes2, train_scores2, test_scores2 = learning_curve(
    knn, X2, y2, cv=gkf, groups=groups, scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
)
plt.figure(figsize=(6, 4.2))
plt.plot(train_sizes2, train_scores2.mean(axis=1), 'o-', label='Training R2')
plt.plot(train_sizes2, test_scores2.mean(axis=1), 'o-', label='Group CV R2 (unseen stores)')
plt.axhline(0, color='gray', linestyle=':')
plt.xlabel('Training set size (rows)')
plt.ylabel('R2')
plt.title('Learning Curve: kNN Regressor (Store one-hot features)')
plt.legend()
plt.tight_layout()
plt.savefig('learning_curve_regressor.png', dpi=150)
plt.close()

X_train2, X_test2, y_train2, y_test2, store_train, store_test = train_test_split(
    X2, y2, groups, test_size=0.2, random_state=RANDOM_STATE
)
knn.fit(X_train2, y_train2)
y_pred2 = knn.predict(X_test2)
err = pd.DataFrame({'store': store_test.values,
                     'abs_err': np.abs(y_test2.values - y_pred2),
                     'actual': y_test2.values})
per_store_mae = err.groupby('store')['abs_err'].mean()
per_store_mean_sales = err.groupby('store')['actual'].mean()
pct_err = per_store_mae / per_store_mean_sales * 100
print("\n=== Per-store error consistency ===")
print(f"MAE range: ${per_store_mae.min():.0f} to ${per_store_mae.max():.0f}")
print(f"As % of store's own mean sales: {pct_err.min():.1f}% to {pct_err.max():.1f}%")

=== Decision Tree, stratified 5-fold CV ===
f1: mean=0.2542 std=0.0085
precision: mean=0.3447 std=0.0077
recall: mean=0.2014 std=0.0088

=== Decision Tree (balanced), stratified 5-fold CV ===
f1: mean=0.1836 std=0.0037
precision: mean=0.1211 std=0.0025
recall: mean=0.3799 std=0.0083

=== Fairlearn MetricFrame by Country ===
                 selection_rate  precision    recall        f1
Country_grouped                                               
EIRE                   0.014033   0.444444  0.292683  0.352941
France                 0.040351   0.673913  0.673913  0.673913
Germany                0.056092   0.795699  0.672727  0.729064
Netherlands            0.009091   0.200000  0.100000  0.133333
Other                  0.030667   0.297297  0.278481  0.287582
Spain                  0.019531   0.400000  0.285714  0.333333
United Kingdom         0.009936   0.315240  0.172769  0.223208

Demographic parity difference: 0.04700076762802939
Equalized odds difference: 0.5739130434782609


C:\Users\ghate\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\ghate\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\ghate\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ghate\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 1039


=== kNN Regressor (store one-hot), GroupKFold by Store ===
r2: mean=-0.8799 std=0.2748
neg_mae: mean=633069.2426 std=110770.2442
neg_rmse: mean=754445.1019 std=105490.2194

=== Per-store error consistency ===
MAE range: $14037 to $177676
As % of store's own mean sales: 4.4% to 12.7%
